In [ ]:
# Configure Auto Loader to read archived HTML and keep its state in the managed volume.
archive_path = 's3://databricks-news-articles/kontrast.at/'
checkpoint_path = '/Volumes/news-analytics/silver/article_ingestion_state/kontrast_articles'
source_files = (spark.readStream
    .format('cloudFiles')
    .option('cloudFiles.format', 'binaryFile')
    .option('cloudFiles.schemaLocation', checkpoint_path)
    .option('pathGlobFilter', '*.html')
    .load(archive_path))
source_files.printSchema()


In [ ]:
# Parse each file batch, prefer the canonical filename, and merge articles into Silver.
def merge_batch(batch_df, batch_id):
    from datetime import datetime
    from pyspark.sql import functions as F
    from pyspark.sql.functions import udf
    from pyspark.sql.window import Window
    from parse_article import parse_article

    if batch_df.isEmpty():
        return

    # Decode HTML and preserve the original S3 path for each article.
    def parse_file(path, content):
        try:
            article = parse_article(content.decode('utf-8', errors='replace'))
        except ValueError as exc:
            raise ValueError(f'{path}: {exc}') from exc
        if article is None:
            return None
        article['published_at'] = datetime.fromisoformat(article['published_at'])
        article['modified_at'] = datetime.fromisoformat(article['modified_at']) if article['modified_at'] else None
        article['source_path'] = path
        return article

    table_name = '`news-analytics`.silver.articles'
    schema = batch_df.sparkSession.table(table_name).schema
    parsed = batch_df.select(udf(parse_file, schema)('path', 'content').alias('article'))
    articles = parsed.filter(F.col('article').isNotNull()).select('article.*')

    # Choose one file per canonical URL within this batch.
    canonical_slug = F.regexp_extract(F.col('canonical_url'), '/([^/]+)/$', 1)
    source_file = F.regexp_extract(F.col('source_path'), '/([^/]+)$', 1)
    ranked = articles.withColumn(
        '_preferred',
        (source_file == F.concat(canonical_slug, F.lit('.html'))).cast('int'),
    )
    rank = Window.partitionBy('canonical_url').orderBy(F.desc('_preferred'), F.asc('source_path'))
    incoming = (ranked
        .withColumn('_rank', F.row_number().over(rank))
        .filter(F.col('_rank') == 1)
        .drop('_preferred', '_rank'))
    incoming.createOrReplaceTempView('incoming_articles')

    # Replace a matching row only when this file wins the same filename rule.
    canonical_file = "concat(regexp_extract(source.canonical_url, '/([^/]+)/$', 1), '.html')"
    source_file_sql = "regexp_extract(source.source_path, '/([^/]+)$', 1)"
    target_file_sql = "regexp_extract(target.source_path, '/([^/]+)$', 1)"
    source_preferred = f'({source_file_sql} = {canonical_file})'
    target_preferred = f'({target_file_sql} = {canonical_file})'
    batch_df.sparkSession.sql(f'''
        MERGE INTO {table_name} AS target
        USING incoming_articles AS source
        ON target.canonical_url = source.canonical_url
        WHEN MATCHED AND (
            ({source_preferred} AND NOT {target_preferred})
            OR ({source_preferred} = {target_preferred}
                AND source.source_path < target.source_path)
        ) THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    ''').collect()


In [ ]:
# Process all currently available HTML files, then stop and save the checkpoint.
query = (source_files.writeStream
    .foreachBatch(merge_batch)
    .option('checkpointLocation', checkpoint_path)
    .trigger(availableNow=True)
    .start())
query.awaitTermination()
print('Auto Loader update finished')


In [ ]:
# Verify Silver URL uniqueness and how many source files Auto Loader tracked.
from pyspark.sql import functions as F

saved_articles = spark.table('`news-analytics`.silver.articles')
display(saved_articles.agg(
    F.count('*').alias('articles'),
    F.countDistinct('canonical_url').alias('distinct_urls'),
))
display(saved_articles.filter(
    F.col('canonical_url') == 'https://kontrast.at/inflation-einfach-erklaert/'
).select('canonical_url', 'source_path'))
display(spark.sql(f"SELECT COUNT(*) AS tracked_files FROM cloud_files_state('{checkpoint_path}')"))
